In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
df_stats = spark.table("workspace.premier_league_silver.team_season_stats")

print(f"Total team-seasons: {df_stats.count()}")
print("\nSeasons available:")
df_stats.groupBy("season").count().orderBy("season").show()

Total team-seasons: 100

Seasons available:
+-------+-----+
| season|count|
+-------+-----+
|2020-21|   20|
|2021-22|   20|
|2022-23|   20|
|2023-24|   20|
|2024-25|   20|
+-------+-----+



In [0]:
# Create previous season lookup - just rename columns with "prev_" prefix
df_prev_lookup = df_stats.select(
    F.col("season").alias("prev_season"),
    F.col("team").alias("prev_team"),
    F.col("position").alias("prev_position"),
    F.col("points").alias("prev_points"),
    F.col("wins").alias("prev_wins"),
    F.col("draws").alias("prev_draws"),
    F.col("losses").alias("prev_losses"),
    F.col("goals_for").alias("prev_goals_for"),
    F.col("goals_against").alias("prev_goals_against"),
    F.col("goal_difference").alias("prev_goal_difference"),
    F.col("home_wins").alias("prev_home_wins"),
    F.col("away_wins").alias("prev_away_wins"),
    F.col("home_goals_for").alias("prev_home_goals_for"),
    F.col("away_goals_for").alias("prev_away_goals_for")
)

print(f"Previous season lookup table: {df_prev_lookup.count()} records")


Previous season lookup table: 100 records


In [0]:
# Define season progression mapping
df_current = df_stats.withColumn(
    "prev_season",
    F.when(F.col("season") == "2021-22", "2020-21")
     .when(F.col("season") == "2022-23", "2021-22")
     .when(F.col("season") == "2023-24", "2022-23")
     .when(F.col("season") == "2024-25", "2023-24")
     .otherwise(None)
)

# Filter out 2020-21 (it has no previous season in our data)
df_current = df_current.filter(F.col("prev_season").isNotNull())

print(f"Current seasons (with previous season available): {df_current.count()} records")
print("\nSeasons included:")
df_current.groupBy("season").count().orderBy("season").show()

Current seasons (with previous season available): 80 records

Seasons included:
+-------+-----+
| season|count|
+-------+-----+
|2021-22|   20|
|2022-23|   20|
|2023-24|   20|
|2024-25|   20|
+-------+-----+



In [0]:
# LEFT JOIN current season with previous season data
df_features = df_current.join(
    df_prev_lookup,
    (df_current.prev_season == df_prev_lookup.prev_season) & 
    (df_current.team == df_prev_lookup.prev_team),
    how="left"
)

# Drop the duplicate prev_season column from the join
df_features = df_features.drop(df_prev_lookup.prev_season).drop(df_prev_lookup.prev_team)

print(f"Records after join: {df_features.count()}")

# Check for teams with no previous season data (promoted teams)
null_count = df_features.filter(F.col("prev_points").isNull()).count()
print(f"\nTeams with NULL prev_points (promoted teams): {null_count}")

if null_count > 0:
    print("\nPromoted teams:")
    df_features.filter(F.col("prev_points").isNull()).select(
        "season", "team", "points"
    ).orderBy("season", "team").show(20)
    

Records after join: 80

Teams with NULL prev_points (promoted teams): 12

Promoted teams:
+-------+-----------------+------+
| season|             team|points|
+-------+-----------------+------+
|2021-22|        Brentford|    46|
|2021-22|          Norwich|    22|
|2021-22|          Watford|    23|
|2022-23|      Bournemouth|    39|
|2022-23|           Fulham|    52|
|2022-23|Nottingham Forest|    38|
|2023-24|          Burnley|    24|
|2023-24|            Luton|    26|
|2023-24|    Sheffield Utd|    16|
|2024-25|          Ipswich|    22|
|2024-25|        Leicester|    25|
|2024-25|      Southampton|    12|
+-------+-----------------+------+



In [0]:
# Calculate league averages for reference
league_avg_points = df_stats.agg(F.avg("points")).collect()[0][0]
league_avg_gf = df_stats.agg(F.avg("goals_for")).collect()[0][0]
league_avg_ga = df_stats.agg(F.avg("goals_against")).collect()[0][0]

print(f"League average points: {league_avg_points:.2f}")
print(f"League average goals for: {league_avg_gf:.2f}")
print(f"League average goals against: {league_avg_ga:.2f}")

# Baseline values for promoted teams
promoted_baseline = {
    "prev_points": 45.0,
    "prev_position": 17.0,
    "prev_wins": 12.0,
    "prev_draws": 9.0,
    "prev_losses": 17.0,
    "prev_goals_for": 47.0,
    "prev_goals_against": 64.0,
    "prev_goal_difference": -17.0,
    "prev_home_wins": 8.0,
    "prev_away_wins": 4.0,
    "prev_home_goals_for": 28.0,
    "prev_away_goals_for": 19.0
}

print(f"\nPromoted team baseline: {promoted_baseline['prev_points']} points, position {promoted_baseline['prev_position']}")

# Create is_promoted flag BEFORE filling nulls
df_features = df_features.withColumn(
    "is_promoted",
    F.when(F.col("prev_points").isNull(), 1).otherwise(0)
)

# Fill nulls with baseline values
df_features = df_features.fillna(promoted_baseline)

# Verify promoted teams were filled
print(f"\nPromoted teams after filling:")
df_features.filter(F.col("is_promoted") == 1).select(
    "season", "team", "is_promoted", "prev_points", "prev_position", "points"
).orderBy("season", "team").show(20)


League average points: 52.67
League average goals for: 55.40
League average goals against: 55.40

Promoted team baseline: 45.0 points, position 17.0

Promoted teams after filling:
+-------+-----------------+-----------+-----------+-------------+------+
| season|             team|is_promoted|prev_points|prev_position|points|
+-------+-----------------+-----------+-----------+-------------+------+
|2021-22|        Brentford|          1|         45|           17|    46|
|2021-22|          Norwich|          1|         45|           17|    22|
|2021-22|          Watford|          1|         45|           17|    23|
|2022-23|      Bournemouth|          1|         45|           17|    39|
|2022-23|           Fulham|          1|         45|           17|    52|
|2022-23|Nottingham Forest|          1|         45|           17|    38|
|2023-24|          Burnley|          1|         45|           17|    24|
|2023-24|            Luton|          1|         45|           17|    26|
|2023-24|    Shef

In [0]:
# Calculate performance metrics
df_features = df_features.withColumn(
    "prev_points_per_game", F.col("prev_points") / 38
).withColumn(
    "prev_win_pct", F.col("prev_wins") / 38
).withColumn(
    "prev_goals_per_game", F.col("prev_goals_for") / 38
).withColumn(
    "prev_goals_conceded_per_game", F.col("prev_goals_against") / 38
).withColumn(
    "prev_home_advantage", 
    (F.col("prev_home_goals_for") - F.col("prev_away_goals_for")) / 38
).withColumn(
    "prev_position_vs_avg", F.col("prev_position") - 10.5
)

# Create position category
df_features = df_features.withColumn(
    "prev_position_category",
    F.when(F.col("prev_position") <= 4, "top4")
     .when(F.col("prev_position") <= 7, "europa")
     .when(F.col("prev_position") <= 14, "mid_table")
     .otherwise("relegation_battle")
)

print("Derived features created successfully")
display(df_features.select(
    "team", "season", "prev_points", "prev_points_per_game", 
    "prev_position_category", "is_promoted"
).show(10))


Derived features created successfully
+--------------------+-------+-----------+--------------------+----------------------+-----------+
|                team| season|prev_points|prev_points_per_game|prev_position_category|is_promoted|
+--------------------+-------+-----------+--------------------+----------------------+-----------+
|     Manchester City|2021-22|         86|   2.263157894736842|                  top4|          0|
|           Liverpool|2021-22|         69|  1.8157894736842106|                  top4|          0|
|             Chelsea|2021-22|         67|   1.763157894736842|                  top4|          0|
|           Tottenham|2021-22|         62|   1.631578947368421|                europa|          0|
|             Arsenal|2021-22|         61|   1.605263157894737|             mid_table|          0|
|   Manchester United|2021-22|         74|  1.9473684210526316|                  top4|          0|
|     West Ham United|2021-22|         65|  1.7105263157894737|        

In [0]:
# Select features and target variables
df_gold = df_features.select(
    # Identifiers
    "season",
    "team",
    
    # Target variables (what we're predicting)
    F.col("points").alias("target_points"),
    F.col("position").alias("target_position"),
    
    # Previous season raw features
    "prev_points",
    "prev_position",
    "prev_position_vs_avg",
    "prev_goal_difference",
    "prev_wins",
    "prev_draws",
    "prev_losses",
    "prev_goals_for",
    "prev_goals_against",
    "prev_home_wins",
    "prev_away_wins",
    "prev_home_goals_for",
    "prev_away_goals_for",
    
    # Derived features
    "prev_points_per_game",
    "prev_win_pct",
    "prev_goals_per_game",
    "prev_goals_conceded_per_game",
    "prev_home_advantage",
    
    # Categorical features
    "prev_position_category",
    
    # Binary flags
    "is_promoted"
)

print(f"Final gold dataset: {df_gold.count()} records")
print(f"Columns: {len(df_gold.columns)}")
print(f"Features: {len(df_gold.columns) - 4} (excluding season, team, and 2 targets)")

# df_gold.show(10)
display(df_gold, raw=10)


Final gold dataset: 80 records
Columns: 24
Features: 20 (excluding season, team, and 2 targets)


season,team,target_points,target_position,prev_points,prev_position,prev_position_vs_avg,prev_goal_difference,prev_wins,prev_draws,prev_losses,prev_goals_for,prev_goals_against,prev_home_wins,prev_away_wins,prev_home_goals_for,prev_away_goals_for,prev_points_per_game,prev_win_pct,prev_goals_per_game,prev_goals_conceded_per_game,prev_home_advantage,prev_position_category,is_promoted
2021-22,Manchester City,93,1,86,1,-9.5,51,27,5,6,83,32,13,14,43,40,2.263157894736842,0.7105263157894737,2.1842105263157894,0.8421052631578947,0.07894736842105263,top4,0
2021-22,Liverpool,92,2,69,3,-7.5,26,20,9,9,68,42,10,10,29,39,1.8157894736842106,0.5263157894736842,1.7894736842105263,1.105263157894737,-0.2631578947368421,top4,0
2021-22,Chelsea,74,3,67,4,-6.5,22,19,10,9,58,36,9,10,31,27,1.763157894736842,0.5,1.5263157894736843,0.9473684210526315,0.10526315789473684,top4,0
2021-22,Tottenham,71,4,62,7,-3.5,23,18,8,12,68,45,10,8,35,33,1.631578947368421,0.47368421052631576,1.7894736842105263,1.1842105263157894,0.05263157894736842,europa,0
2021-22,Arsenal,69,5,61,8,-2.5,16,18,7,13,55,39,8,10,24,31,1.605263157894737,0.47368421052631576,1.4473684210526316,1.0263157894736843,-0.18421052631578946,mid_table,0
2021-22,Manchester United,58,6,74,2,-8.5,29,21,11,6,73,44,9,12,38,35,1.9473684210526316,0.5526315789473685,1.9210526315789473,1.1578947368421053,0.07894736842105263,top4,0
2021-22,West Ham United,56,7,65,6,-4.5,15,19,8,11,62,47,10,9,32,30,1.7105263157894737,0.5,1.631578947368421,1.236842105263158,0.05263157894736842,europa,0
2021-22,Leicester,52,8,66,5,-5.5,18,20,6,12,68,50,9,11,34,34,1.736842105263158,0.5263157894736842,1.7894736842105263,1.3157894736842106,0.0,europa,0
2021-22,Brighton & Hove Albion,51,9,41,16,5.5,-6,9,14,15,40,46,4,5,22,18,1.0789473684210527,0.23684210526315788,1.0526315789473684,1.2105263157894737,0.10526315789473684,relegation_battle,0
2021-22,Wolves,51,10,45,13,2.5,-16,12,9,17,36,52,7,5,21,15,1.1842105263157894,0.3157894736842105,0.9473684210526315,1.368421052631579,0.15789473684210525,mid_table,0


In [0]:
print("=== DATA QUALITY CHECKS ===\n")

# Check 1: Record counts by season
print("Records per season:")
display(df_gold.groupBy("season").count().orderBy("season"))

# Check 2: Null values
print("\nNull value check:")
for col in df_gold.columns:
    null_count = df_gold.filter(F.col(col).isNull()).count()
    if null_count > 0:
        print(f"  {col}: {null_count} nulls")

# Check 3: Summary statistics
print("\nSummary statistics for key features:")
display(
    df_gold.select(
        "prev_points", 
        "prev_position",
        "prev_goal_difference",
        "target_points"
    ).summary("count", "mean", "min", "max")
)

# Check 4: Promoted teams distribution
print("\nPromoted teams by season:")
display(df_gold.groupBy("season", "is_promoted").count().orderBy("season", "is_promoted"))

# Check 5: Position category distribution
print("\nTeams by previous position category:")
display(df_gold.groupBy("prev_position_category").count())

print("\n Data quality validation complete!")


=== DATA QUALITY CHECKS ===

Records per season:


season,count
2021-22,20
2022-23,20
2023-24,20
2024-25,20



Null value check:

Summary statistics for key features:


summary,prev_points,prev_position,prev_goal_difference,target_points
count,80,80,80,80
mean,55.5875,10.2,3.15,52.625
min,36,1,-37,12
max,93,17,73,93



Promoted teams by season:


season,is_promoted,count
2021-22,0,17
2021-22,1,3
2022-23,0,17
2022-23,1,3
2023-24,0,17
2023-24,1,3
2024-25,0,17
2024-25,1,3



Teams by previous position category:


prev_position_category,count
relegation_battle,24
mid_table,28
europa,12
top4,16



 Data quality validation complete!


In [0]:
# Create gold schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.premier_league_gold")

# Save as Delta table
df_gold.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.premier_league_gold.training_features"
)

print(" Gold table created: workspace.premier_league_gold.training_features")
print(f"   Total records: {df_gold.count()}")
print(f"   Seasons: 2021-22 through 2024-25")
print(f"   Features: {len(df_gold.columns) - 4}")


 Gold table created: workspace.premier_league_gold.training_features
   Total records: 80
   Seasons: 2021-22 through 2024-25
   Features: 20


In [0]:
# Sample training data

# Top 5 teams from 2024-25 season
display(
    df_gold.filter(F.col("season") == "2024-25")
    .orderBy("target_position")
    .select("team", "target_position", "target_points", "prev_points", "prev_position", "is_promoted"),
    raw=5
)

# Promoted teams across all seasons
display(
    df_gold.filter(F.col("is_promoted") == 1)
    .select("season", "team", "prev_points", "target_points", "target_position")
    .orderBy("season", "team"),
    raw=20
)

# Example returning team (Manchester City)
display(
    df_gold.filter(F.col("team") == "Manchester City")
    .select("season", "prev_points", "target_points", "prev_position", "target_position", "is_promoted")
    .orderBy("season")
)

team,target_position,target_points,prev_points,prev_position,is_promoted
Liverpool,1,84,82,3,0
Arsenal,2,74,89,2,0
Manchester City,3,71,91,1,0
Chelsea,4,69,63,6,0
Newcastle United,5,66,60,7,0
Aston Villa,6,66,68,4,0
Nottingham Forest,7,65,36,17,0
Brighton & Hove Albion,8,61,48,11,0
Bournemouth,9,56,48,13,0
Brentford,10,56,39,16,0


season,team,prev_points,target_points,target_position
2021-22,Brentford,45,46,13
2021-22,Norwich,45,22,20
2021-22,Watford,45,23,19
2022-23,Bournemouth,45,39,15
2022-23,Fulham,45,52,10
2022-23,Nottingham Forest,45,38,16
2023-24,Burnley,45,24,19
2023-24,Luton,45,26,18
2023-24,Sheffield Utd,45,16,20
2024-25,Ipswich,45,22,19


season,prev_points,target_points,prev_position,target_position,is_promoted
2021-22,86,93,1,1,0
2022-23,93,89,1,1,0
2023-24,89,91,1,1,0
2024-25,91,71,1,3,0
